In [ ]:
import pandas as pd

df = pd.read_csv(r'hasta_kayitlari.csv')

### Bölüm 1: Veriyi Tanımak

In [ ]:
df.head(8)

In [ ]:
df.tail(4)

In [ ]:
df.info()

bmi, sistolik ve maliyet_tl verilerinde sırasıyla 48, 15 ve 22 veri eksik.

In [ ]:
df.rename(columns={"maliyet_tl":"maliyet"},inplace=True)
df.rename(columns={"yatis_gun":"yatis_suresi"},inplace=True)
df.head(3)

### Bölüm 2: Veri Temizleme

In [ ]:
print("Tekrar eden veriler silinmeden önce uzunluk: ", len(df))
df.duplicated().sum()

In [ ]:
df=df.drop_duplicates()
print("Tekrar eden veriler silindikten sonra uzunluk: ", len(df))

Sütunlardaki eksik veri miktarlarını listeleyelim

In [ ]:
print(df.isna().sum())

bmi, sistolik ve maliyet_tl verilerinde sırasıyla 48, 15 ve 22 veri eksik.

In [ ]:
df['bmi'] = df['bmi'].fillna(df['bmi'].median())
print(df.isna().sum())

In [ ]:
df['sistolik_kb'] = df['sistolik_kb'].fillna(df['sistolik_kb'].median())
print(df.isna().sum())

In [ ]:
df = df.dropna(subset=['maliyet'])
print(df.isna().sum())

In [ ]:
len(df)

### Bölüm 3: Filtreleme ve Yeni Sütun Oluşturma

In [ ]:
df[(df.sehir=="İstanbul") & (df.yas>50)]

İstanbul'da yaşayan ve yaşı 50'den büyük 124 kişi vardır

In [ ]:
sigortasiz_yatis = df[(df.sigorta=="Yok") & (df.yatis_suresi>7)]
sigortasiz_yatis['maliyet'].mean()

Sigortası olmayan ve 7 günden fazla yatış yapan hastaların ortalama maliyeti 21769.53

In [ ]:
df['gunluk_maliyet'] = df['maliyet'] / df['yatis_suresi']
df['yas_grubu'] = [
    'genç' if 18 < i <= 35 
    else 'orta yaşlı' if 35 < i < 60 
    else 'yaşlı'
    for i in df['yas']
]
df.head(3)

In [ ]:
df[df['bolum'] == 'Kardiyoloji'].sort_values('maliyet', ascending=False).head(10)

Kardiyoloji bölümünde en çok maliyete sahip hastalarından ilk 10 kişiyi seçtiğimizde büyük çoğunluğu sigortalıdır (8 sigortalı, 2 sigortasız). Sigortalıların çoğu da SGK'lıdır.

### Bölüm 4: Özet İstatistikleri 

In [ ]:
df.describe().T

In [ ]:
df['sehir'].value_counts()

In [ ]:
df['sehir'].value_counts(normalize=True)

Hasta yoğunluğu olarak en kalabalık 3 şehir sırasıyla İstanbul, Ankara ve İzmir'dir. 
Bu şehirlerin diğer şehirlere kıyasla yoğunluk oranı sırasıyla %28.9, %16.8 ve %15.4 dür.

In [ ]:
df['bolum'].value_counts()

In [ ]:
df['bolum'].value_counts(normalize=True)

Bölümlerin hasta yoğunlukları birbirlerine yakın olmakla birlikte en yoğun iki bölüm sırasıyla Kardiyoloji ve Dahiliye bölümleridir. Bu bölümlerin hasta sayısının toplama oranı sırasıyla %18.0 ve %17.5'dir.

In [ ]:
df.groupby("bolum")[["yas", "yatis_suresi", "maliyet"]].agg(["mean", "max", "count", "sum"]).round(2).T

Her bölüm için ortalama ve maksimum maliyet yatış süresi ve yaş bulunmuştur.

In [ ]:
df.groupby("yas_grubu")[["bmi"]].mean().round(2)

Yaş ile BMI arasında doğrusal korelasyon gözlenmiştir. Yaş arttıkça ortalama BMI artmaktadır

In [ ]:
df.groupby("sigorta")[["yatis_suresi", "maliyet"]].mean().round(2).T

Sigortasıs hastaların maliyeti sigortalılara kıyasla daha fazladır.

In [ ]:
df[df["sigorta"] == "Yok"].groupby("bolum").size().sort_values(ascending=False)

En çok sigortasız hasta Kardiyoloji bölümünde bulunmaktadır.